In [ ]:
import os
import pathlib
import pandas as pd

In [ ]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

# Lecture du fichier source

In [ ]:
dataframe_export = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export.xlsx",
    )
)
dataframe_export

# Nettoyage

In [ ]:
# Suppression des lignes entièrement vides et des lignes de métadonnées
# (ex : pied de page "Filtres appliqués" présent dans certains exports filtrés)
dataframe_export.dropna(how="all", inplace=True)

mask_metadata = dataframe_export.apply(
    lambda row: row.astype(str).str.contains("Filtres appliqués", case=False, na=False).any(),
    axis=1
)
dataframe_export = dataframe_export[~mask_metadata].reset_index(drop=True)
dataframe_export

In [ ]:
# Renommage des colonnes utiles
dataframe_export = dataframe_export.rename(
    columns={
        "DISTRIBUTEUR": "Distributeur",
        "CODE PRODUIT": "Code produit",
        "UF": "Unité transaction (UF)",
    }
)
dataframe_export

# Construction de la table de correspondance

In [ ]:
# Extraction des combinaisons uniques (Distributeur, Code produit, Unité transaction)
# Chaque combinaison = une ligne dans la table de correspondance
dataframe_correspondance = (
    dataframe_export[["Distributeur", "Code produit", "Unité transaction (UF)"]]
    .drop_duplicates()
    .dropna(subset=["Distributeur", "Code produit", "Unité transaction (UF)"])
    .sort_values(["Distributeur", "Code produit"])
    .reset_index(drop=True)
)

# Colonnes à remplir manuellement par le client
# Unité accord : unité dans laquelle l'accord est exprimé (ex : UVC, Colis, KG)
# Facteur de conversion : nombre d'unités accord par unité de transaction (ex : 1 colis = 40 UVC → facteur = 40)
dataframe_correspondance["Unité accord"] = "UVC"
dataframe_correspondance["Facteur de conversion"] = 1

dataframe_correspondance

# Export

In [ ]:
dataframe_correspondance.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "table_correspondance_uvc.xlsx",
    ),
    index=False
)